# 第6回　確率分布と標本／中心極限定理
## ―― なぜ「一部」から「全体」を語れるのか

統計学Ⅰ（B）　／　北星学園大学

今日から **推測統計**。手元のデータ（標本）から、見ていない全体（母集団）を推し量る。注目は ――

> 全部見なくていい。**標本の平均は、母集団の平均の周りに、きれいな形で散らばる。**

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)
except Exception:
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan

# 今日の「母集団」：北辰大の一般家庭の世帯年収（極端な富裕2世帯は別扱いとして除く）
母集団 = df.loc[df["世帯年収万円"] < 5000, "世帯年収万円"].values
rng = np.random.default_rng(2026)
print("準備OK　母集団の人数:", len(母集団))

---
## フック：なぜ「一部」で「全体」が分かる？

- 視聴率は、全国数千万世帯のうち**たった数百世帯**を調べて出している。
- 味噌汁の味見は、鍋全体を飲まなくても**ひと匙**で分かる。

なぜ一部で全体を語れるのか。今日はその数学的な裏づけ＝**中心極限定理**を、自分の目で見る。

---
## 1. 母集団と標本

- **母集団**：本当に知りたい全体（例：全国民の年収）
- **標本**：実際に調べる一部（例：選ばれた数百人）

今日の母集団は「北辰大の一般家庭の世帯年収」。まず母集団の形を見よう。**右に歪んでいて、正規分布ではない。**

In [ ]:
mu = 母集団.mean()
sigma = 母集団.std()
print(f"母平均 μ = {mu:.0f} 万円")
print(f"母標準偏差 σ = {sigma:.0f} 万円")
print(f"歪度 = {pd.Series(母集団).skew():.2f}（0でない＝歪んでいる）")

plt.figure(figsize=(7,3.5))
plt.hist(母集団, bins=40, color="#80cbc4", edgecolor="white")
plt.axvline(mu, color="#1565c0", lw=2, label=f"母平均 {mu:.0f}")
plt.xlabel("世帯年収（万円）"); plt.ylabel("人数"); plt.legend(); plt.title("母集団は正規分布ではない（右に歪む）")
plt.show()

---
## 2. 標本を1つ取ってみる（無作為抽出）

母集団から **n=30人** をランダムに選び、その標本平均を出す。実行するたびに少しずつ違う値になる（運で変わる）。

In [ ]:
標本 = rng.choice(母集団, size=30, replace=False)
print("選ばれた30人の年収:", np.round(標本).astype(int))
print(f"\nこの標本の平均 = {標本.mean():.0f} 万円（母平均 {mu:.0f} に近いが、ぴったりではない）")

---
## 3. 標本平均を「何度も」取ると…（標本分布）

1回だけだと運に左右される。では **n=30 の標本取り→平均を2000回くりかえす**と、標本平均たちはどんな形に散らばる？

母集団は歪んでいた。標本平均の分布も歪むだろうか？

In [ ]:
標本平均たち = np.array([rng.choice(母集団, 30).mean() for _ in range(2000)])
print(f"標本平均の平均 = {標本平均たち.mean():.0f}（母平均 {mu:.0f} とほぼ一致）")
print(f"標本平均のばらつき(SD) = {標本平均たち.std():.0f}")
print(f"標本平均の分布の歪度 = {pd.Series(標本平均たち).skew():.2f}（母集団の歪度より小さい＝正規に近づいた）")

plt.figure(figsize=(7,3.5))
plt.hist(標本平均たち, bins=30, color="#e8503a", edgecolor="white")
plt.axvline(mu, color="#1565c0", lw=2, label=f"母平均 {mu:.0f}")
plt.xlabel("標本平均（n=30）"); plt.ylabel("回数"); plt.legend()
plt.title("母集団は歪んでいたのに、標本平均の分布は釣鐘型（正規）に")
plt.show()

**これが中心極限定理(CLT)。** 母集団がどんな形でも、標本平均をたくさん集めると、その分布は**正規分布に近づく**。だから「一部」から「全体の平均」を語れる。

---
## 4. 標本サイズ n を変えると ―― 分布が「締まる」

n を 5・30・100 と増やして、標本平均の分布を比べる。**n が大きいほど、母平均の周りにギュッと締まる**はず。

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.3), sharex=True)
for i, n in enumerate([5, 30, 100]):
    means = np.array([rng.choice(母集団, n).mean() for _ in range(2000)])
    ax[i].hist(means, bins=30, color="#80cbc4", edgecolor="white")
    ax[i].axvline(mu, color="#1565c0", lw=2)
    ax[i].set_title(f"n={n}\n標本平均SD={means.std():.0f}  σ/√n={sigma/np.sqrt(n):.0f}")
    ax[i].set_xlabel("標本平均")
plt.suptitle("n が大きいほど、標本平均は母平均の近くに締まる"); plt.tight_layout(); plt.show()

標本平均のばらつき（**標準誤差 SE**）は **σ/√n** にぴったり一致する。n を4倍にするとSEは半分。だから「精度を2倍にするには4倍のデータが要る」。

$$ 標準誤差\ SE = \frac{母標準偏差\ \sigma}{\sqrt{n}} $$

---
## 5. 大事な区別：「標本そのもの」≠「標本平均の分布」

よくある誤解：「標本を取ると正規分布になる」。**ちがう。** 正規に近づくのは**標本平均の分布**であって、標本そのものは母集団と同じ形（歪んだまま）だ。並べて確認しよう。

In [ ]:
一つの標本 = rng.choice(母集団, 1000)                         # 標本そのもの（1000人）
標本平均の分布 = np.array([rng.choice(母集団, 30).mean() for _ in range(2000)])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].hist(一つの標本, bins=40, color="#80cbc4", edgecolor="white")
ax[0].set_title("標本そのもの（1000人）→ 歪んだまま"); ax[0].set_xlabel("年収")
ax[1].hist(標本平均の分布, bins=30, color="#e8503a", edgecolor="white")
ax[1].set_title("標本平均(n=30)の分布 → 正規に近い"); ax[1].set_xlabel("標本平均")
plt.tight_layout(); plt.show()

---
## 6. ただし「無作為（ランダム）」が大前提

CLTが効くのは、標本が**母集団から偏りなく・独立に**選ばれているとき。

- もし「声の大きい人」「答えやすい人」ばかり選ぶと、いくら数を増やしても**偏ったまま**。これは数を増やしても直らない（標本数の問題ではなく、抽出の偏りの問題）。
- みんなが**互いに影響し合って**答えをそろえる（＝空気を読む）と、標本の**独立性**が壊れ、推測の前提が崩れる。

> ⚠️ **数の多さは、偏りを直さない**
>
> 偏った集め方をしたデータは、何万件あっても母集団を正しく代表しない。**『たくさん集めた』は『正しく集めた』ではない。**

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 母集団 / 標本 | 知りたい全体 / 実際に調べる一部 |
| 中心極限定理(CLT) | 母集団がどんな形でも、標本平均の分布は正規に近づく |
| 標準誤差 SE = σ/√n | 標本平均のばらつき。n を増やすと小さくなる |
| 標本 ≠ 標本平均 | 正規に近づくのは「標本平均の分布」。標本そのものは母集団の形のまま |
| 無作為抽出 | CLTの大前提。偏った抽出は数を増やしても直らない |

> **一部から全体を語れるのは、標本平均がCLPによって正規分布に従うから。**
> ただし「無作為に集めた」ことが大前提 ―― 数の多さは偏りを直さない。

次回からは、この標本平均の散らばり（SE）を使って、母平均を**区間**で推定していく。

**課題（Moodle）**：CLTのシミュレーション結果を読み、「n を増やすと標本平均の分布に何が起きたか」を説明する。